In [17]:
from gwtorch.modules.neural_net import SimpleCNN
from gwtorch.modules.ml_utils import load_all_data
import torch
from gwtorch.modules.ml_utils import get_model

In [49]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = 'cpu'

In [50]:
data  = '/home/qubit/Templates/tmp/data_new'
csv_path  = '/home/qubit/Templates/tmp/results/data_new'

In [51]:
num_channels, train_loader, val_loader, test_loader, class_weights = load_all_data(data, batch_size=32, meta_csv_path=csv_path)

In [52]:
cnn_model = SimpleCNN(input_channels=num_channels)
cnn_model = cnn_model.to(device=device)
print(cnn_model)

SimpleCNN(
  (conv1): Conv2d(4, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=100352, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=3, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
)


In [53]:
%%timeit

cnn_model.eval()

with torch.no_grad():
    for i, (inputs, labels, *_) in enumerate(test_loader):
        inputs = inputs.to(device=device)
        outputs = cnn_model(inputs)
        

1.77 s ± 15.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [55]:
vit_model = get_model('ViT', num_channels=num_channels, device=device)
print(vit_model)

ViT(
  (to_patch_embedding): Sequential(
    (0): Rearrange('b c (h p1) (w p2) -> b (h w) (p1 p2 c)', p1=32, p2=32)
    (1): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
    (2): Linear(in_features=4096, out_features=128, bias=True)
    (3): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (transformer): Linformer(
    (net): SequentialSequence(
      (layers): ModuleList(
        (0-11): 12 x ModuleList(
          (0): PreNorm(
            (fn): LinformerSelfAttention(
              (to_q): Linear(in_features=128, out_features=128, bias=False)
              (to_k): Linear(in_features=128, out_features=128, bias=False)
              (to_v): Linear(in_features=128, out_features=128, bias=False)
              (dropout): Dropout(p=0.2, inplace=False)
              (to_out): Linear(in_features=128, out_features=128, bias=True)
            )
            (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          )
          (1): PreNorm(
            (fn): Fe

In [56]:
%%timeit

vit_model.eval()

with torch.no_grad():
    for i, (inputs, labels, *_) in enumerate(test_loader):
        inputs = inputs.to(device=device)
        outputs = vit_model(inputs)
        

1.75 s ± 20.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [57]:
len(test_loader.dataset)

1350